In [ ]:
# Source Code 2 
# Script to download GeoTIFF images for specified bounding boxes.

# pip install segment-geospatial # Library to install the required dependencies.

import csv # Used for reading CSV files.
import os # Used for creating directories and file path operations.
import time # Used for adding delays between retry attempts.
from samgeo import tms_to_geotiff # Used to download and convert map tiles.

# Create a new folder to save downloaded GeoTIFF images
new_folder_path = "E:/wi/0-1000000"

try:
    os.makedirs(new_folder_path, exist_ok=True)
    print(f"Folder '{new_folder_path}' created successfully.")
except Exception as e:
    print(f"An error occurred while creating the folder: {e}")

# Create a list to store the points' latitude, longitude, and bounding box sizes
csv_file_name = "E:/wi/wi_bounding_boxes.csv"

points_list = []

with open(csv_file_name, 'r') as csvfile:
    csvreader = csv.DictReader(csvfile) # Use csv.DictReader to access columns by their names
    for row in csvreader:
        x = float(row["x"])
        y = float(row["y"])
        north = float(row["north"])
        south = float(row["south"])
        east = float(row["east"])
        west = float(row["west"])
        points_list.append((x, y, north, south, east, west))

# Set the start and end indices for the slice
start_index = 0 # Adjust this to the desired starting index
end_index = 1000000 # Adjust this to the desired ending index

# Set the maximum number of retries
max_retries = 5

# Load the last successful index from a file
try:
    with open('last_successful_index.txt', 'r') as file:
        last_successful_index = int(file.read().strip())
except FileNotFoundError:
    last_successful_index = start_index

# Function to download GeoTIFF image with retry mechanism
def download_geotiff_with_retry(output_path, bbox):
    retries = 0
    while retries < max_retries:
        try:
            # Geospatial operation - Download and convert map tiles to a GeoTIFF image
            tms_to_geotiff(output=output_path, bbox=bbox, zoom=19, source="ROADMAP", overwrite=True)
            return True # Download successful
        except Exception as e:
            print(f"Download failed: {e}. Retrying...")
            retries += 1
            time.sleep(5) # Add a delay before retrying
            print("Max retries exceeded. Download unsuccessful.")
            return False

# Loop through each point and extract GeoTIFF image
for i, (x, y, north, south, east, west) in enumerate(points_list[last_successful_index:end_index],
                                                     start=last_successful_index):
    # Determine the bounding box for the current point
    bbox = [west, south, east, north]
    
    # Specify the output path for each GeoTIFF image (customize as needed)
    output_path = f"E:/wi/0-1000000/wi{i}_x{x}_y{y}.tif"
    
    # Attempt to download with retry mechanism
    if download_geotiff_with_retry(output_path, bbox):
        # Update the last successful index
        last_successful_index = i
        # Save the last successful index to a file
        with open('last_successful_index.txt', 'w') as file:
            file.write(str(last_successful_index))
            # Check if we have reached the end of the slice, and break the loop if necessary
            if i == end_index - 1:
                break